# 01b - Pré-processamento dos pedidos rejeitados

Pré-processamento do ficheiro *rejected* (Fase 2 do enunciado, *Pre-processing*). Os pedidos rejeitados não entram na regressão nem na classificação. São usados em duas análises que comparam aceites e rejeitados:
- o **clustering de estados** (taxas de aprovação, montantes pedidos e *risk scores* por estado);
- os **testes de hipóteses** "um *risk score* mais baixo implica mais rejeição" e "o montante pedido difere entre aceites e rejeitados".

**Ordem dos passos** (a mesma do `01_preprocessamento.ipynb`): primeiro só o que não depende de nenhuma estatística dos dados, ou seja, a verificação estrutural e a **partição temporal** com os cortes da Fase 1. A partir daí, todas as análises usam **só o desenvolvimento**.

O enunciado pede três cuidados específicos com este ficheiro, tratados nas secções indicadas:
- `Risk_Score` = 0 conta como valor em falta;
- o DTI vem em texto com "%" (secção 4);
- há artefactos de introdução de dados (ex.: DTI de 4042.11%).

**Antes de correr:** é preciso ter corrido `00_conversao_dados.ipynb`. O ficheiro *rejected* ocupa cerca de 3,5 GB em memória.

In [ ]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.exclusions import ExclusionLog
from lending.parsing import parse_iso_date
from lending.partition import TemporalPartitioner
from lending.rejected import (REJECTED_SCHEMA, REJECTED_SCHEMA_RENAMED, REJECTED_UNITS,
                              RejectedCleaner)
from lending.schema import SchemaReport
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Ficheiro de origem e verificação estrutural

No *accepted* havia linhas que não eram empréstimos (totais e separadores das exportações). Aqui verifica-se o mesmo: se as colunas são as 9 esperadas e se há linhas vazias ou com uma estrutura diferente.

In [ ]:
rejected = PickleStore.load(config.REJECTED_RAW_PKL, expected_type=pd.DataFrame)
print(f"rejected: {len(rejected):,} linhas x {rejected.shape[1]} colunas")

assert list(rejected.columns) == list(REJECTED_SCHEMA), "colunas diferentes das esperadas"
print("Colunas iguais às do schema:", list(rejected.columns))

# Linhas sem nenhum valor e linhas sem data do pedido (não podiam ser atribuídas a uma partição)
print(f"Linhas totalmente vazias: {rejected.isna().all(axis=1).sum()}")
print(f"Linhas sem Application Date: {rejected['Application Date'].isna().sum()}")

Não há linhas estruturais: todas as linhas são pedidos, todas têm data. Não é preciso nenhuma remoção antes da partição.

## 2. Partição temporal

Os pedidos rejeitados seguem os **mesmos cortes** dos aceites (`desenho_split.md`), usando a data do pedido (`Application Date`):

| Partição | Período | Utilização |
|---|---|---|
| Antes do primeiro aceite | 2007-05 | Fora do estudo: não há empréstimos aceites neste mês, por isso não há com que comparar |
| Desenvolvimento | 2007-06 a 2014-12 | Clustering de estados, testes de hipóteses |
| Teste final | 2015 | Não usado: os rejeitados não entram nos modelos supervisionados |
| Reservado | 2016-2018 | Não usado na Parte 1 |

Converter a data é parsing determinístico (um formato fixo, `AAAA-MM-DD`), por isso pode ser feito antes da partição. Os cortes são meses fixos em `config.REJECTED_PARTITIONS`, nunca a ordem das linhas.

In [ ]:
application_date = parse_iso_date(rejected["Application Date"])
print(f"Application Date: {application_date.min():%Y-%m-%d} a {application_date.max():%Y-%m-%d}")

partitioner = TemporalPartitioner(config.REJECTED_PARTITIONS)
partitioner.summary(application_date)

In [ ]:
# Pedidos por ano e partição: mostra o crescimento dos pedidos depois de 2012
pd.crosstab(application_date.dt.year.rename("ano"),
            partitioner.assign(application_date).rename("partição"))

In [ ]:
log = ExclusionLog("rejected -> desenvolvimento")
sizes = partitioner.summary(application_date)["rows"]

log.record(
    step="Pedidos anteriores ao primeiro aceite",
    kind="partição",
    rule="Application Date em 2007-05",
    reason="O primeiro empréstimo aceite é de 2007-06: sem aceites no mesmo mês, não há comparação",
    rows_before=len(rejected),
    rows_removed=int(sizes["before_accepted"]),
)
log.record(
    step="Separar o teste final",
    kind="partição",
    rule="Application Date de 2015-01 a 2015-12",
    reason="Mesmo corte dos aceites; os rejeitados do teste não são usados (não entram nos modelos)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["test"]),
)
log.record(
    step="Separar o período reservado",
    kind="partição",
    rule="Application Date de 2016-01 a 2018-12",
    reason="Não usado na Parte 1 (mesmo corte dos aceites)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["reserved"]),
)

# Só o desenvolvimento passa para memória; o ficheiro completo é libertado
rejected_dev = partitioner.split(rejected, application_date, names=["development"])["development"]
assert len(rejected_dev) == log.rows_after == sizes["development"]

del rejected, application_date
_ = gc.collect()

log.to_frame()

## 3. Guardar a partição

Tal como no *accepted*, a partição é guardada **ainda em texto**, sem nenhum valor alterado:
- `rejected_dev_raw.pkl`: pedidos rejeitados do desenvolvimento;
- `rejected_dev_exclusions.pkl`: o registo de exclusões, que os passos seguintes continuam.

O teste e o período reservado não são guardados: não são usados e continuam disponíveis em `rejected_raw.pkl`.

In [ ]:
PickleStore.save(rejected_dev, config.REJECTED_DEV_RAW_PKL)
PickleStore.save(log, config.REJECTED_DEV_LOG_PKL)
print(f"Desenvolvimento: {len(rejected_dev):,} linhas -> {config.REJECTED_DEV_RAW_PKL.name}")
print(f"Registo:         {log} -> {config.REJECTED_DEV_LOG_PKL.name}")

## 4. Conversão de tipos (parsing determinístico)

Cada coluna tem um tipo declarado à mão em `REJECTED_SCHEMA` (`src/lending/rejected.py`) e é convertida pelo mesmo `SchemaParser` do *accepted*:

| Coluna original | Nome usado | Tipo | Conversão |
|---|---|---|---|
| Amount Requested | `amount_requested` | number | "1000.0" → 1000.0 |
| Application Date | `application_date` | date | "2007-05-26" → data |
| Loan Title | `title` | text | espaços nas pontas retirados; só espaços → em falta |
| Risk_Score | `risk_score` | number | "693.0" → 693.0 |
| Debt-To-Income Ratio | `dti` | percent | "38.64%" → 38.64 (escala 0–100, como o `dti` do *accepted*) |
| Zip Code | `zip_code` | category | "481xx" (3 primeiros dígitos, como no *accepted*) |
| State | `addr_state` | category | sigla do estado |
| Employment Length | `emp_length` | emp_length | "< 1 year" → 0 … "10+ years" → 10 (mesma tabela do *accepted*) |
| Policy Code | `policy_code` | number | "0.0" → 0.0 |

As colunas passam a ter o **nome da coluna equivalente do *accepted***, para que comparar aceites e rejeitados seja direto. Um valor com formato inesperado dá erro em vez de passar a valor em falta sem ninguém reparar.

In [ ]:
cleaner = RejectedCleaner()
rejected_dev_parsed = cleaner.parse(rejected_dev)

# Controlo: só texto vazio pode passar a valor em falta (o parser dá erro noutros casos)
cleaner.summary(rejected_dev, rejected_dev_parsed)

In [ ]:
# Antes e depois, nas mesmas linhas (amostra aleatória com semente, só para ver a conversão)
sample = rejected_dev.sample(5, random_state=config.RANDOM_SEED).index
display(rejected_dev.loc[sample])
display(rejected_dev_parsed.loc[sample])

In [ ]:
# Tabelas de conversão: cada texto corresponde a um só número
emp_check = (pd.DataFrame({"texto": rejected_dev["Employment Length"],
                           "anos": rejected_dev_parsed["emp_length"]})
             .value_counts(dropna=False).rename("pedidos").reset_index()
             .sort_values("anos", na_position="last"))
assert emp_check["texto"].dropna().is_unique
display(emp_check)

# DTI: todos os valores preenchidos terminavam em "%" e passaram a número
print(f"DTI: {rejected_dev['Debt-To-Income Ratio'].notna().sum():,} preenchidos -> "
      f"{rejected_dev_parsed['dti'].notna().sum():,} números")

In [ ]:
# A partir daqui trabalha-se com o desenvolvimento já convertido
# (a versão em texto continua guardada em rejected_dev_raw.pkl)
rejected_dev = rejected_dev_parsed
del rejected_dev_parsed
_ = gc.collect()

rejected_dev.dtypes.rename("dtype").to_frame()

## 5. Schema e unidades

O `SchemaReport` (o mesmo do *accepted*) descreve cada coluna: tipo, unidade, intervalo observado e categorias mais frequentes. A tabela fica em `docs/02_preprocessamento/schema_rejected.md`.

In [ ]:
schema_report = SchemaReport(schema=REJECTED_SCHEMA_RENAMED, units=REJECTED_UNITS)
schema_description = schema_report.describe(rejected_dev)

out = schema_report.write_markdown(schema_description,
                                   config.DOCS_DIR / "02_preprocessamento" / "schema_rejected.md",
                                   dataset="rejected, desenvolvimento", n_rows=len(rejected_dev),
                                   notebook="01b_rejeitados.ipynb")
print("Escrito:", out.relative_to(config.PROJECT_ROOT))
schema_description